# UTS Machine Learning — Polynomial Regression untuk Prediksi Harga Properti

> **Instruktur:** Ida Bagus Kresna Sudiatmika, S.Kom., M.T.

Notebook ini disusun untuk memenuhi seluruh *requirements* UTS: EDA → Preprocessing → Polynomial (deg 1–5) → Linear/Ridge/Lasso → Evaluasi (R²,MSE,RMSE,MAE,MAPE) → Learning Curves → Regularisasi (α grid) → Feature Importance → Model Selection (CV) → Prediksi data baru → Simpan artefak.

Jika Anda menggunakan Google Colab, jalankan setiap sel dari atas ke bawah.


## 0. Setup & Imports

In [ ]:
import os, math, json, joblib, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (7, 5)
os.makedirs("artifacts", exist_ok=True)


## 1. Data Loading / Generation

In [ ]:
# Ubah path ini bila Anda menggunakan dataset lain
CSV_PATH = "data/properties.csv"
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError("Dataset tidak ditemukan. Pastikan `data/properties.csv` tersedia.")

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
display(df.head())


## 2. EDA — Exploratory Data Analysis

In [ ]:
# 2A. Statistical summary
display(df.describe(include="all"))

# 2B. Histograms
for col in df.columns:
    plt.figure()
    df[col].hist(bins=30, edgecolor='black')
    plt.title(f"Histogram: {col}")
    plt.xlabel(col); plt.ylabel("Count")
    plt.show()

# 2C. Scatter vs target
target = "harga_juta"
features = [c for c in df.columns if c != target]
for col in features:
    plt.figure()
    plt.scatter(df[col], df[target], alpha=0.6)
    plt.title(f"Scatter: {col} vs {target}")
    plt.xlabel(col); plt.ylabel(target)
    plt.show()

# 2D. Correlation heatmap
corr = df.corr(numeric_only=True)
print(corr.round(3))
plt.figure()
plt.imshow(corr, cmap='coolwarm', interpolation='nearest')
plt.title("Correlation Heatmap")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.tight_layout()
plt.show()

# 2E. Outlier check (simple IQR on target)
Q1, Q3 = df[target].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df[target] < lower) | (df[target] > upper)]
print(f"Outliers (simple IQR rule) on {target}: {len(outliers)} rows")


## 3. Preprocessing — Train/Test Split + Scaling + Missing Handling

In [ ]:
# 70/30 split
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Simple missing handling (fill with median)
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test  = X_test.fillna(X_train.median(numeric_only=True))  # use train statistics

# Use StandardScaler by default (can switch to MinMaxScaler below)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, "artifacts/scaler.joblib")
print("Scaler saved to artifacts/scaler.joblib")


## 4. Helper Functions (Metrics, Training, Evaluation Tables)

In [ ]:
def regression_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    # avoid division by zero
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
    return {"R2": r2, "MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE_%": mape}

def train_poly_model(Xtr, ytr, Xte, yte, degree=1, model_type="linear", alpha=1.0):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    Xtr_poly = poly.fit_transform(Xtr)
    Xte_poly = poly.transform(Xte)

    if model_type == "linear":
        model = LinearRegression()
    elif model_type == "ridge":
        model = Ridge(alpha=alpha, random_state=42)
    elif model_type == "lasso":
        model = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    else:
        raise ValueError("Unknown model_type")

    model.fit(Xtr_poly, ytr)

    ytr_pred = model.predict(Xtr_poly)
    yte_pred = model.predict(Xte_poly)

    train_m = regression_metrics(ytr, ytr_pred)
    test_m  = regression_metrics(yte, yte_pred)
    return model, poly, train_m, test_m

def as_table(rows):
    dfm = pd.DataFrame(rows)
    display(dfm.sort_values(by=["Degree","Model","Alpha"]).reset_index(drop=True))
    return dfm


## 5. Polynomial Features (Degree 1–5) & Linear/Ridge/Lasso

In [ ]:
results = []
saved_models = {}

degrees = [1,2,3,4,5]
ridge_alphas = [0.1, 1.0, 10.0]
lasso_alphas = [0.1, 1.0, 10.0]

for deg in degrees:
    # Linear
    model, poly, tr, te = train_poly_model(X_train_scaled, y_train, X_test_scaled, y_test, degree=deg, model_type="linear")
    results.append({"Degree":deg, "Model":"Linear", "Alpha":0, **{f"train_{k}":v for k,v in tr.items()}, **{f"test_{k}":v for k,v in te.items()}})
    saved_models[(deg,"linear",0)] = (model, poly)

    # Ridge
    for a in ridge_alphas:
        model, poly, tr, te = train_poly_model(X_train_scaled, y_train, X_test_scaled, y_test, degree=deg, model_type="ridge", alpha=a)
        results.append({"Degree":deg, "Model":"Ridge", "Alpha":a, **{f"train_{k}":v for k,v in tr.items()}, **{f"test_{k}":v for k,v in te.items()}})
        saved_models[(deg,"ridge",a)] = (model, poly)

    # Lasso
    for a in lasso_alphas:
        model, poly, tr, te = train_poly_model(X_train_scaled, y_train, X_test_scaled, y_test, degree=deg, model_type="lasso", alpha=a)
        results.append({"Degree":deg, "Model":"Lasso", "Alpha":a, **{f"train_{k}":v for k,v in tr.items()}, **{f"test_{k}":v for k,v in te.items()}})
        saved_models[(deg,"lasso",a)] = (model, poly)

results_df = as_table(results)

# Tampilkan jumlah fitur untuk setiap degree (tanpa bias)
for deg in degrees:
    pf = PolynomialFeatures(degree=deg, include_bias=False)
    pf.fit(np.zeros((1, X_train.shape[1])))
    print(f"Degree {deg} -> jumlah fitur: {pf.n_output_features_}")


## 6. Learning Curves (Over/Underfitting Analysis)

In [ ]:
def plot_learning_curve_for(degree):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    Xtr_poly = poly.fit_transform(X_train_scaled)
    est = LinearRegression()
    train_sizes, train_scores, test_scores = learning_curve(est, Xtr_poly, y_train, cv=5, scoring='r2', train_sizes=np.linspace(0.2, 1.0, 5), random_state=42)
    plt.figure()
    plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train R²')
    plt.plot(train_sizes, test_scores.mean(axis=1), marker='s', label='CV R²')
    plt.title(f"Learning Curve — Degree {degree} (Linear)")
    plt.xlabel("Train size"); plt.ylabel("R²"); plt.legend(); plt.grid(True); plt.show()

for deg in [1,2,3,4,5]:
    plot_learning_curve_for(deg)


## 7. Visualization — Predicted vs Actual, Residuals, Polynomial Curve (2 fitur) & R² vs Degree

In [ ]:
# Ambil salah satu model terbaik berdasarkan test_R2
best_row = results_df.sort_values(by="test_R2", ascending=False).iloc[0]
key = (int(best_row["Degree"]), best_row["Model"].lower(), float(best_row["Alpha"]))
best_model, best_poly = saved_models[key]

Xte_poly = best_poly.transform(X_test_scaled)
y_pred = best_model.predict(Xte_poly)

# Predicted vs Actual
plt.figure()
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--')
plt.title("Predicted vs Actual (Test)")
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.show()

# Residual plot
residuals = y_test - y_pred
plt.figure()
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, linestyle='--')
plt.title("Residuals vs Predicted (Test)")
plt.xlabel("Predicted"); plt.ylabel("Residual")
plt.show()

# R2 vs Degree (pakai Linear model)
deg_r2 = results_df[results_df["Model"]=="Linear"].groupby("Degree")["test_R2"].max()
plt.figure()
plt.plot(deg_r2.index, deg_r2.values, marker='o')
plt.title("R² (Test) vs Degree — Linear")
plt.xlabel("Degree"); plt.ylabel("R² (Test)")
plt.grid(True)
plt.show()

# Polynomial curve (2 fitur penting): fix other features at median
f1, f2 = "luas_bangunan_m2", "luas_tanah_m2"
med = X_train.median(numeric_only=True)
grid1 = np.linspace(X[f1].min(), X[f1].max(), 80)
grid2 = np.linspace(X[f2].min(), X[f2].max(), 80)

# 3D-like via mesh projection to 2D heatmap of predicted price
g1, g2 = np.meshgrid(grid1, grid2)
Xgrid = pd.DataFrame({col: med[col] for col in X.columns})
Xgrid = pd.concat([Xgrid] * g1.size, axis=0, ignore_index=True)

Xgrid[f1] = g1.ravel()
Xgrid[f2] = g2.ravel()

Xgrid_scaled = scaler.transform(Xgrid)
Xgrid_poly = best_poly.transform(Xgrid_scaled)
pred_grid = best_model.predict(Xgrid_poly).reshape(g1.shape)

plt.figure()
plt.contourf(g1, g2, pred_grid, levels=20)
plt.colorbar(label="Predicted harga_juta")
plt.xlabel(f1); plt.ylabel(f2)
plt.title(f"Surface (proj) harga_juta vs {f1} & {f2} — Model Terbaik")
plt.show()


## 8. Regularization Analysis — Ridge vs Lasso (α grid)

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
deg = 3  # contoh fokus degree 3

r2_ridge, r2_lasso = [], []
for a in alphas:
    m, p, tr, te = train_poly_model(X_train_scaled, y_train, X_test_scaled, y_test, degree=deg, model_type="ridge", alpha=a)
    r2_ridge.append(te["R2"])
    m, p, tr, te = train_poly_model(X_train_scaled, y_train, X_test_scaled, y_test, degree=deg, model_type="lasso", alpha=a)
    r2_lasso.append(te["R2"])

plt.figure()
plt.semilogx(alphas, r2_ridge, marker='o', label="Ridge (Test R²)")
plt.semilogx(alphas, r2_lasso, marker='s', label="Lasso (Test R²)")
plt.title("R² vs Alpha — Degree 3")
plt.xlabel("Alpha (log)"); plt.ylabel("R² (Test)"); plt.legend(); plt.grid(True)
plt.show()

best_ridge_alpha = alphas[int(np.argmax(r2_ridge))]
best_lasso_alpha = alphas[int(np.argmax(r2_lasso))]
print("Alpha optimal (ridge,l ecolass o) ≈", best_ridge_alpha, best_lasso_alpha)


## 9. Feature Importance (Koefisien) & Eliminasi Fitur oleh Lasso

In [ ]:
def feature_names_after_poly(feature_names, degree):
    pf = PolynomialFeatures(degree=degree, include_bias=False)
    pf.fit(np.zeros((1,len(feature_names))))
    return pf.get_feature_names_out(feature_names)

# Ambil koefisien dari model terbaik
feature_names = list(X.columns)
names_poly = feature_names_after_poly(feature_names, int(best_row["Degree"]))
coefs = pd.Series(best_model.coef_, index=names_poly).sort_values(key=np.abs, ascending=False)
display(coefs.head(20))

# Bar plot top 20 absolute coefficients
top = coefs.head(20)
plt.figure()
top_abs = top.abs()
top_abs.sort_values(ascending=True).plot(kind='barh')
plt.title("Top 20 |Koefisien| — Model Terbaik")
plt.xlabel("|Coefficient|")
plt.tight_layout(); plt.show()

# Lasso elimination demo (degree=3, alpha best from grid above)
lasso = Lasso(alpha=best_lasso_alpha, random_state=42, max_iter=10000)
pf = PolynomialFeatures(degree=3, include_bias=False)
Xtr_poly = pf.fit_transform(X_train_scaled)
lasso.fit(Xtr_poly, y_train)
names = pf.get_feature_names_out(feature_names)
zeros = (lasso.coef_ == 0)
eliminated = np.array(names)[zeros]
print(f"Fitur yang dieliminasi oleh Lasso (alpha={best_lasso_alpha}): {len(eliminated)}")
print(eliminated[:20])


## 10. Model Selection via Cross-Validation (K-Fold) & Penyimpanan Model Terbaik

In [ ]:
def cv_score_for(deg, model_kind, alpha=0.0, k=5):
    pf = PolynomialFeatures(degree=deg, include_bias=False)
    Xtr_poly = pf.fit_transform(X_train_scaled)

    if model_kind == "linear":
        est = LinearRegression()
    elif model_kind == "ridge":
        est = Ridge(alpha=alpha, random_state=42)
    elif model_kind == "lasso":
        est = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    else:
        raise ValueError

    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(est, Xtr_poly, y_train, cv=kf, scoring='r2')
    return scores.mean(), scores.std()

# Search best by highest Test R², then lowest RMSE as tiebreaker
results_df["rank_key"] = list(zip(-results_df["test_R2"], results_df["test_RMSE"]))
best_row = results_df.sort_values(by=["rank_key"]).iloc[0]
key = (int(best_row["Degree"]), best_row["Model"].lower(), float(best_row["Alpha"]))
best_model, best_poly = saved_models[key]

# CV score for the chosen
cv_mean, cv_std = cv_score_for(int(best_row["Degree"]), best_row["Model"].lower(), float(best_row["Alpha"]) if best_row["Model"]!="Linear" else 0.0, k=5)

print("Best by Test performance:")
display(best_row)
print(f"CV mean R² (k=5): {cv_mean:.4f} ± {cv_std:.4f}")

# Save best model & polynomial features
joblib.dump(best_model, "artifacts/best_model.joblib")
joblib.dump(best_poly, "artifacts/best_poly.joblib")
print("Saved best_model.joblib and best_poly.joblib in artifacts/")


## 11. Final Prediction Function + Confidence Interval (Bootstrap)

In [ ]:
def predict_with_ci(model, poly, scaler, X_df, n_boot=300, alpha=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    Xs = scaler.transform(X_df)
    Xp = poly.transform(Xs)
    preds = model.predict(Xp)

    # simple bootstrap of residuals from train set
    # fit residuals
    Xtr_poly = poly.transform(X_train_scaled)
    res = y_train.values - model.predict(Xtr_poly)
    res_sample = rng.choice(res, size=(n_boot, len(X_df)), replace=True)
    boot = preds + res_sample
    lo = np.percentile(boot, 100*alpha/2, axis=0)
    hi = np.percentile(boot, 100*(1-alpha/2), axis=0)
    return preds, lo, hi

# Test with 5 unseen samples
sample_new = pd.DataFrame({
    "luas_tanah_m2":[80, 150, 220, 300, 450],
    "luas_bangunan_m2":[60, 120, 180, 250, 380],
    "kamar_tidur":[2,3,3,4,5],
    "umur_bangunan_tahun":[5,10,8,15,3],
    "jarak_ke_pusat_kota_km":[4,8,10,6,2],
})
preds, lo, hi = predict_with_ci(best_model, best_poly, joblib.load("artifacts/scaler.joblib"), sample_new)

result_pred = sample_new.copy()
result_pred["pred_harga_juta"] = preds
result_pred["ci_low"] = lo
result_pred["ci_high"] = hi
display(result_pred.round(2))

# Simpan tabel prediksi
result_pred.to_csv("artifacts/sample_predictions.csv", index=False)
print("Saved artifacts/sample_predictions.csv")


## 12. BONUS — Polynomial Regression dari Nol (Gradient Descent)

In [ ]:
class PolyRegScratch:
    def __init__(self, degree=2, lr=1e-3, n_iter=10000, reg=None, alpha=0.0):
        self.degree = degree
        self.lr = lr
        self.n_iter = n_iter
        self.reg = reg  # None | "ridge" | "lasso"
        self.alpha = alpha

    def _poly(self, X):
        pf = PolynomialFeatures(self.degree, include_bias=True)
        return pf.fit_transform(X)

    def fit(self, X, y):
        Xp = self._poly(X)
        n, d = Xp.shape
        self.w = np.zeros(d)
        y = y.astype(float)

        for _ in range(self.n_iter):
            yhat = Xp @ self.w
            grad = (2/n) * (Xp.T @ (yhat - y))
            # simple regularization
            if self.reg == "ridge":
                grad += 2 * self.alpha * self.w
            elif self.reg == "lasso":
                grad += self.alpha * np.sign(self.w)

            self.w -= self.lr * grad
        self.d_ = d
        return self

    def predict(self, X):
        Xp = self._poly(X)
        return Xp @ self.w

# Train & compare (degree=2) on scaled data
model_scratch = PolyRegScratch(degree=2, lr=5e-4, n_iter=4000, reg="ridge", alpha=1e-3)
model_scratch.fit(X_train_scaled, y_train.values)
y_pred_scratch = model_scratch.predict(X_test_scaled)
print("Scratch model (deg=2, ridge-like) test R²:", r2_score(y_test, y_pred_scratch))


---
## 13. Ringkasan & Insight Otomatis
Sel ini membuat ringkasan singkat untuk membantu penyusunan laporan.

In [ ]:
# Simple summary based on results_df
top = results_df.sort_values(by="test_R2", ascending=False).head(5)[["Degree","Model","Alpha","test_R2","test_RMSE","test_MAPE_%"]]
print("Top-5 Model (berdasarkan R² Test tertinggi):")
display(top)

best = results_df.sort_values(by=["rank_key"]).iloc[0]
print("\nModel Terbaik (berdasarkan R² Test tinggi & RMSE rendah):")
display(best[["Degree","Model","Alpha","test_R2","test_RMSE","test_MAPE_%"]])

print("\nSaran awal: gunakan derajat yang memberikan generalisasi baik (R² test tinggi, gap train-test kecil), "
      "dan pertimbangkan Ridge/Lasso bila derajat tinggi menyebabkan overfitting.")        
